# Chuyển `customer_turns` thành audio bằng MiniMax TTS

Notebook Google Colab này đọc **một file scenario JSON**, lấy `calls.<call_id>.customer_turns`, rồi tạo **một file WAV cho mỗi cuộc gọi** bằng MiniMax Synchronous Text-to-Speech API.

Trước khi chạy:

1. Chạy ô cài đặt thư viện và mount Google Drive.
2. Lưu API key với tên `MINIMAX_API_KEY` trong **Colab Secrets**, hoặc thêm `MINIMAX_API_KEY=...` vào `/content/.env`. Không ghi key trực tiếp vào notebook.
3. Chọn `SCENARIO_FILE`, kiểm tra phần xem trước, rồi đổi `RUN_TTS = True`.

> Mặc định `RUN_TTS = False` để Run All không vô tình sử dụng quota. API tham chiếu: https://platform.minimax.io/docs/api-reference/speech-t2a-http

In [ ]:
%pip install -q requests python-dotenv

In [ ]:
import json
import os
import re
from pathlib import Path

import requests
from dotenv import load_dotenv
from IPython.display import Audio, display

## 1. Mount Google Drive và chọn file đầu vào

Đổi `SCENARIO_FILE` nếu file của bạn có tên hoặc vị trí khác. Các đường dẫn phải là đối tượng `Path`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content")
DATASET_DIR = Path("/content/drive/MyDrive/scenarios_complete dataset")

# Thay tên file tại đây nếu cần.
SCENARIO_FILE = DATASET_DIR / "SC-001.json"
OUTPUT_DIR = Path("/content")

if not SCENARIO_FILE.is_file():
    raise FileNotFoundError(f"Không tìm thấy file: {SCENARIO_FILE}")

print("Input :", SCENARIO_FILE)
print("Output:", OUTPUT_DIR)

## 2. Đọc và kiểm tra `customer_turns`

In [ ]:
def load_customer_turns(json_path: Path) -> tuple[str, dict[str, list[str]]]:
    """Đọc scenario và trả về (scenario_id, {call_id: customer_turns})."""
    with json_path.open("r", encoding="utf-8") as file:
        payload = json.load(file)

    if not isinstance(payload, dict):
        raise ValueError("Nội dung JSON phải là một object.")

    scenario_id = str(payload.get("scenario_id") or json_path.stem)
    calls = payload.get("calls")
    if not isinstance(calls, dict) or not calls:
        raise ValueError("Không tìm thấy object 'calls' hoặc 'calls' đang rỗng.")

    turns_by_call: dict[str, list[str]] = {}
    for call_id, call_data in calls.items():
        if not isinstance(call_data, dict):
            raise ValueError(f"{call_id} phải là một object.")

        turns = call_data.get("customer_turns")
        if not isinstance(turns, list):
            raise ValueError(f"{call_id}.customer_turns phải là một list.")
        if not all(isinstance(turn, str) for turn in turns):
            raise ValueError(f"Mọi phần tử trong {call_id}.customer_turns phải là chuỗi.")

        cleaned_turns = [turn.strip() for turn in turns if turn.strip()]
        if not cleaned_turns:
            raise ValueError(f"{call_id}.customer_turns không có câu hợp lệ.")
        turns_by_call[str(call_id)] = cleaned_turns

    return scenario_id, turns_by_call


scenario_id, customer_turns_by_call = load_customer_turns(SCENARIO_FILE)
print(f"Scenario: {scenario_id}")
for call_id, turns in customer_turns_by_call.items():
    print(f"\n{call_id} ({len(turns)} câu):")
    for index, text in enumerate(turns, start=1):
        print(f"  {index}. {text}")

## 3. Cấu hình MiniMax TTS

- `CALL_IDS = None`: xử lý tất cả các cuộc gọi; ví dụ `["call_1"]` để chỉ xử lý một call.
- `VOICE_ID` mặc định là voice tiếng Việt có sẵn của MiniMax.
- Các lượt khách trong cùng một call được nối bằng pause marker của MiniMax.
- `OVERWRITE = False`: bỏ qua WAV đã tồn tại.

In [ ]:
API_URL = "https://api.minimax.io/v1/t2a_v2"
MODEL = "speech-2.8-hd"
VOICE_ID = "Vietnamese_kindhearted_girl"
LANGUAGE_BOOST = "Vietnamese"

SPEED = 1.0
VOLUME = 1.0
PITCH = 0
SAMPLE_RATE = 32000
BITRATE = 128000
AUDIO_FORMAT = "wav"
CHANNELS = 1
TURN_PAUSE_SECONDS = 0.6
REQUEST_TIMEOUT_SECONDS = 180

CALL_IDS = None  # None = tất cả; ví dụ: ["call_1"]
OVERWRITE = False
RUN_TTS = False  # Đổi thành True sau khi đã kiểm tra nội dung phía trên.

load_dotenv(PROJECT_ROOT / ".env")
MINIMAX_API_KEY = os.getenv("MINIMAX_API_KEY")

# Nếu chưa có trong biến môi trường/.env, thử lấy từ Colab Secrets.
if not MINIMAX_API_KEY:
    try:
        from google.colab import userdata

        MINIMAX_API_KEY = userdata.get("MINIMAX_API_KEY")
    except Exception:
        MINIMAX_API_KEY = None

print("Đã tìm thấy MINIMAX_API_KEY:", bool(MINIMAX_API_KEY))

In [ ]:
def safe_filename(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", value).strip("._") or "audio"


def build_call_text(text_chunks: list[str]) -> str:
    if not 0.01 <= TURN_PAUSE_SECONDS <= 99.99:
        raise ValueError("TURN_PAUSE_SECONDS phải nằm trong khoảng 0.01–99.99.")

    pause_marker = f"<#{TURN_PAUSE_SECONDS:.2f}#>"
    text = pause_marker.join(chunk.strip() for chunk in text_chunks if chunk.strip())
    if not text:
        raise ValueError("Không có nội dung hợp lệ để tạo audio.")
    if len(text) >= 10_000:
        raise ValueError(
            f"Nội dung có {len(text):,} ký tự; MiniMax yêu cầu dưới 10.000 ký tự/request."
        )
    return text


def synthesize_call(
    api_key: str,
    scenario_id: str,
    call_id: str,
    text_chunks: list[str],
    output_dir: Path,
) -> tuple[Path, int, dict]:
    """Gọi MiniMax TTS và tạo một WAV cho toàn bộ customer_turns của một call."""
    text = build_call_text(text_chunks)
    payload = {
        "model": MODEL,
        "text": text,
        "stream": False,
        "language_boost": LANGUAGE_BOOST,
        "output_format": "hex",
        "voice_setting": {
            "voice_id": VOICE_ID,
            "speed": SPEED,
            "vol": VOLUME,
            "pitch": PITCH,
        },
        "audio_setting": {
            "sample_rate": SAMPLE_RATE,
            "bitrate": BITRATE,
            "format": AUDIO_FORMAT,
            "channel": CHANNELS,
        },
    }
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }

    response = requests.post(
        API_URL,
        headers=headers,
        json=payload,
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    try:
        body = response.json()
    except ValueError as exc:
        raise RuntimeError(
            f"MiniMax trả về HTTP {response.status_code} nhưng không phải JSON."
        ) from exc

    if not response.ok:
        raise RuntimeError(f"MiniMax HTTP {response.status_code}: {body}")

    base_resp = body.get("base_resp") or {}
    if base_resp.get("status_code") != 0:
        raise RuntimeError(
            "MiniMax API lỗi "
            f"{base_resp.get('status_code')}: {base_resp.get('status_msg')} "
            f"(trace_id={body.get('trace_id')})"
        )

    data = body.get("data")
    if not isinstance(data, dict) or not data.get("audio"):
        raise RuntimeError(
            f"MiniMax không trả về audio (trace_id={body.get('trace_id')})."
        )

    try:
        audio = bytes.fromhex(data["audio"])
    except ValueError as exc:
        raise RuntimeError("Audio MiniMax trả về không phải chuỗi hex hợp lệ.") from exc
    if not audio:
        raise RuntimeError("MiniMax trả về audio rỗng.")

    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / (
        f"{safe_filename(scenario_id)}_{safe_filename(call_id)}_minimax_customer.{AUDIO_FORMAT}"
    )
    output_path.write_bytes(audio)

    metadata = {
        "trace_id": body.get("trace_id"),
        "text_characters": len(text),
        "extra_info": body.get("extra_info") or {},
    }
    return output_path, len(audio), metadata

## 4. Gọi MiniMax và lưu WAV

Ô này chỉ gọi API khi `RUN_TTS = True`. Kết quả được lưu trong `/content/<scenario_id>/`, kèm `manifest_minimax.json`.

In [ ]:
if not RUN_TTS:
    print("Chưa gọi API. Hãy đổi RUN_TTS = True trong ô cấu hình khi sẵn sàng.")
else:
    if not MINIMAX_API_KEY:
        raise EnvironmentError(
            "Thiếu MINIMAX_API_KEY trong Colab Secrets, /content/.env hoặc biến môi trường."
        )

    if CALL_IDS is None:
        selected_call_ids = list(customer_turns_by_call)
    else:
        if isinstance(CALL_IDS, str):
            raise TypeError("CALL_IDS phải là list, ví dụ ['call_1'], không phải chuỗi.")
        selected_call_ids = list(dict.fromkeys(CALL_IDS))
        missing = [call_id for call_id in selected_call_ids if call_id not in customer_turns_by_call]
        if missing:
            raise KeyError(f"CALL_IDS không tồn tại trong file: {missing}")

    scenario_output_dir = OUTPUT_DIR / safe_filename(scenario_id)
    results = []
    generated_files = []

    for call_id in selected_call_ids:
        expected_path = scenario_output_dir / (
            f"{safe_filename(scenario_id)}_{safe_filename(call_id)}_minimax_customer.{AUDIO_FORMAT}"
        )
        if expected_path.exists() and not OVERWRITE:
            print(f"Bỏ qua file đã tồn tại: {expected_path.name}")
            results.append(
                {
                    "call_id": call_id,
                    "status": "skipped_existing",
                    "output_file": str(expected_path),
                }
            )
            continue

        try:
            output_path, byte_count, metadata = synthesize_call(
                api_key=MINIMAX_API_KEY,
                scenario_id=scenario_id,
                call_id=call_id,
                text_chunks=customer_turns_by_call[call_id],
                output_dir=scenario_output_dir,
            )
            generated_files.append(output_path)
            results.append(
                {
                    "call_id": call_id,
                    "status": "generated",
                    "turn_count": len(customer_turns_by_call[call_id]),
                    "bytes": byte_count,
                    "output_file": str(output_path),
                    **metadata,
                }
            )
            print(f"Đã ghi {byte_count:,} bytes: {output_path}")
        except Exception as exc:
            results.append(
                {
                    "call_id": call_id,
                    "status": "error",
                    "error": f"{type(exc).__name__}: {exc}",
                }
            )
            print(f"Lỗi ở {call_id}: {type(exc).__name__}: {exc}")

    scenario_output_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = scenario_output_dir / "manifest_minimax.json"
    manifest = {
        "source_file": str(SCENARIO_FILE),
        "scenario_id": scenario_id,
        "api_url": API_URL,
        "model": MODEL,
        "voice_id": VOICE_ID,
        "language_boost": LANGUAGE_BOOST,
        "audio_format": AUDIO_FORMAT,
        "results": results,
    }
    manifest_path.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print("Manifest:", manifest_path)

    if generated_files:
        print("Nghe thử file vừa tạo đầu tiên:")
        display(Audio(filename=str(generated_files[0])))